# URI, URL, and URN - Rust

All 10 Rust examples from [docs/uri.md](https://platob.github.io/yggdryl/uri/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::Uri;

let uri = Uri::from_str("HTTPS://example.test/archive/report.tar.gz?q=1#summary")?;

assert_eq!(uri.to_string(), "https://example.test/archive/report.tar.gz?q=1#summary");
assert_eq!(uri.scheme().as_str(), "https");
assert_eq!(uri.authority().as_str(), "example.test");
assert_eq!(uri.path().as_str(), "/archive/report.tar.gz");
assert_eq!(uri.query(), Some("q=1"));
assert_eq!(uri.fragment(), Some("summary"));
assert_eq!(uri.file_name(), Some("report.tar.gz"));

## Canonical on arrival

In [ ]:
use yggdryl::Uri;

// The scheme lowercases and percent escapes uppercase.
let uri = Uri::from_str("HTTPS://example.test/caf%c3%a9.csv")?;
assert_eq!(uri.to_string(), "https://example.test/caf%C3%A9.csv");

// Backslashes in a `file:` hierarchy are separators, not data.
let windows = Uri::from_str(r"file:///C:\Users\Ada\report.parquet")?;
assert_eq!(windows.to_string(), "file:///C:/Users/Ada/report.parquet");

// A string with no usable scheme is a filesystem path.
assert_eq!(
    Uri::from_str("/var/lib/data.arrow")?.to_string(),
    "file:///var/lib/data.arrow"
);
assert_eq!(
    Uri::from_str("data/ticks.csv")?.to_string(),
    "file:data/ticks.csv"
);

// A colon after the first separator is data, so this is not a scheme.
let stamped = Uri::from_str("/data/2026-08-16T00:00:00/part.parquet")?;
assert_eq!(stamped.scheme().as_str(), "file");

// Canonical output re-parses to the same value.
assert_eq!(Uri::from_str(&uri.to_string())?, uri);

## Path segments

In [ ]:
use yggdryl::Uri;

let uri = Uri::from_str("https://example.test/archive/2026/report.tar.gz")?;
let segments: Vec<&str> = uri.path_segments().collect();

assert_eq!(segments, ["archive", "2026", "report.tar.gz"]);
assert_eq!(uri.path().segment_len(), 3);
assert_eq!(uri.path().get_segment(0), Some("archive"));
assert!(uri.path().contains_segment("2026"));

// `&Uri` iterates its own segments, borrowing them from the URI.
let mut visited: Vec<&str> = Vec::new();
for segment in &uri {
    visited.push(segment);
}
assert_eq!(visited, segments);

## Compound filenames

In [ ]:
use yggdryl::Uri;

let mut uri = Uri::from_str("https://example.test/archive/report.tar.gz?q=1#part")?;

assert_eq!(uri.file_name(), Some("report.tar.gz"));
assert_eq!(uri.stem(), Some("report.tar"));
assert_eq!(uri.extension(), Some("gz"));
assert_eq!(uri.extensions().collect::<Vec<_>>(), ["tar", "gz"]);

// Renaming touches the filename and nothing else.
uri.set_stem("renamed")?;
assert_eq!(uri.to_string(), "https://example.test/archive/renamed.gz?q=1#part");
uri.set_extensions(["csv", "gz"])?;
assert_eq!(uri.to_string(), "https://example.test/archive/renamed.csv.gz?q=1#part");
assert!(uri.remove_extension());
assert!(uri.clear_extensions());
assert_eq!(uri.to_string(), "https://example.test/archive/renamed?q=1#part");

// A rejected name changes nothing.
let unchanged = uri.to_string();
assert!(uri.set_file_name("bad/name").is_err());
assert_eq!(uri.to_string(), unchanged);

## The media type is in the name

In [ ]:
use yggdryl::{MediaType, MimeType, Uri};

let mut uri = Uri::from_str("https://example.test/report.csv.gz.zst?q=1#part")?;

// The final suffix is the MIME type; the whole chain is the media type.
assert_eq!(uri.mime_type(), MimeType::ZSTD);
let media = uri.media_type();
assert_eq!(media.base(), &MimeType::CSV);
assert_eq!(media.encodings(), &[MimeType::GZIP, MimeType::ZSTD]);

// Setting a MIME type rewrites the final suffix.
uri.set_mime_type(MimeType::JSON)?;
assert_eq!(uri.to_string(), "https://example.test/report.csv.gz.json?q=1#part");

// Setting a media type rewrites the whole chain.
let encoded = MediaType::from_parts(MimeType::CSV, [MimeType::GZIP, MimeType::ZSTD])?;
uri.set_media_type(encoded)?;
assert_eq!(uri.to_string(), "https://example.test/report.csv.gz.zst?q=1#part");

// A MIME type with no preferred extension cannot name a file.
let unchanged = uri.to_string();
let custom = MimeType::from_str("application/vnd.example")?;
assert!(uri.set_mime_type(custom).is_err());
assert_eq!(uri.to_string(), unchanged);

## URL and URN

In [ ]:
use yggdryl::{Uri, Url, Urn};

let uri = Uri::from_str("https://example.test/a/data.json?raw=true")?;
let url = Url::from_uri(uri.clone())?;
assert_eq!(url.authority().as_str(), "example.test");
assert_eq!(Uri::from(&url), uri);

let urn = Urn::from_str("URN:ISBN:9780131103627")?;
assert_eq!(urn.to_string(), "urn:isbn:9780131103627");
assert_eq!(urn.namespace(), "isbn");
assert_eq!(urn.namespace_specific(), "9780131103627");
assert_eq!(urn.authority().as_str(), "");

// Each refuses what it is not.
assert!(urn.to_uri().to_url().is_err());
assert!(Urn::from_uri(uri).is_err());
assert!(Url::from_str("mailto:user@example.test").is_err());
assert!(Url::from_str("https:///missing-authority").is_err());

## Platform paths

In [ ]:
use std::path::PathBuf;
use yggdryl::{Uri, Url};

// Drive and UNC detection is textual, so it behaves the same on every host.
let uri = Uri::try_from(PathBuf::from(r"C:\Users\Ada Lovelace\report.parquet"))?;
assert_eq!(uri.to_string(), "file:///C:/Users/Ada%20Lovelace/report.parquet");
assert_eq!(uri.authority().as_str(), "");
assert_eq!(uri.file_name(), Some("report.parquet"));

// And back, with the escapes decoded.
let path = PathBuf::try_from(&uri)?;
assert_eq!(path, PathBuf::from("C:/Users/Ada Lovelace/report.parquet"));
assert_eq!(Uri::try_from(path)?, uri);

// A UNC share puts the server in the authority.
let unc = Uri::from_path(r"\\server\share\prices\ticks.csv")?;
assert_eq!(unc.to_string(), "file://server/share/prices/ticks.csv");
assert_eq!(unc.authority().as_str(), "server");
assert_eq!(unc.to_path()?, PathBuf::from("//server/share/prices/ticks.csv"));

// Only a `file:` identifier has a path at all.
assert!(Url::from_str("https://example.test/data.csv")?.to_path().is_err());

## Walking the path

In [ ]:
use yggdryl::{UriPath, Url};

let url = Url::from_str("https://example.test/a/b/c?q=1#frag")?;

// `joinpath` composes the way a shell `cd` does; everything else survives.
let joined = url.joinpath("../d")?;
assert_eq!(joined.to_string(), "https://example.test/a/b/d?q=1#frag");

// `parts` is the sequence of names the path actually addresses.
assert_eq!(url.parts(), ["a", "b", "c"]);

// `parents` climbs to the root and never yields the value itself.
let parents: Vec<String> = url
    .parents()
    .map(|value| value.path().as_str().to_owned())
    .collect();
assert_eq!(parents, ["/a/b", "/a", "/"]);
assert_eq!(url.parent().unwrap().path().as_str(), "/a/b");

// `..` past an absolute root is clamped, matching filesystem semantics.
assert_eq!(UriPath::from_str("/../../a")?.parts(), ["a"]);
// A relative path has no root to clamp against, so `..` is kept.
assert_eq!(UriPath::from_str("../../a")?.parts(), ["..", "..", "a"]);

## What the scheme decides

In [ ]:
use yggdryl::{MimeType, Uri, Url};

// The port belongs to the scheme, not to the authority text.
assert_eq!(Url::from_str("https://example.test")?.default_port(), Some(443));
assert_eq!(Uri::from_str("postgres://host/db")?.default_port(), Some(5432));
assert_eq!(Uri::from_str("s3://bucket/key")?.default_port(), None);

let root = std::env::temp_dir().join(format!("yggdryl-doc-uri-{}", std::process::id()));
let _ = std::fs::remove_dir_all(&root);
std::fs::create_dir_all(&root)?;
std::fs::write(root.join("ticks.csv"), b"symbol\n")?;

// `join_path` is `Path::join` for URLs: one segment per component.
let folder = Url::try_from(root.as_path())?;
assert!(folder.is_local());
assert!(folder.is_dir());
assert_eq!(folder.local_mime_type(), MimeType::DIRECTORY);

let file = folder.join_path("ticks.csv")?;
assert!(file.exists());
assert!(file.is_file());
assert_eq!(file.local_mime_type(), MimeType::CSV);

// An existing file the name cannot identify is still a file.
std::fs::write(root.join("MANIFEST"), b"")?;
assert_eq!(folder.join_path("MANIFEST")?.local_mime_type(), MimeType::FILE);

// A remote URL answers without a round trip: it is simply not local.
let remote = Url::from_str("https://example.test/ticks.csv")?;
assert!(!remote.is_local());
assert!(!remote.exists());
assert_eq!(remote.local_mime_type(), MimeType::CSV);

let _ = std::fs::remove_dir_all(&root);

## Patterns and partitions

In [ ]:
use yggdryl::Url;

let pattern = Url::from_str("file:///lake/trades/year=2024/**/*.parquet")?;
assert!(pattern.is_glob());
assert!(pattern.is_recursive_glob());

// A glob decomposes into the deepest fixed location and the rest.
let (root, rest) = pattern.glob_parts()?;
assert_eq!(root.to_string(), "file:///lake/trades/year=2024");
assert_eq!(rest.as_deref(), Some("**/*.parquet"));

// Matching follows the `.gitignore` rule.
let part = Url::from_str("file:///lake/trades/year=2024/month=01/part-0.parquet")?;
assert!(part.matches_glob("*.parquet"));
assert!(part.matches_glob("lake/**/part-?.parquet"));
assert!(!part.matches_glob("lake/*.parquet"));
assert!(part.matches_glob_under(&root, "**/*.parquet"));

// The directory names are the partition columns.
assert_eq!(part.hive_partition("month").as_deref(), Some("01"));
assert_eq!(
    part.hive_partitions(),
    vec![("year".to_owned(), "2024".to_owned()), ("month".to_owned(), "01".to_owned())]
);